# Day03：淘宝商品数据Pandas探索

**姓名：** 吴泽昊

本Notebook由每名学生独立完成，并提交到个人GitHub仓库。

> 请完成所有TODO和检查点。不要覆盖原始数据文件。

## 实验目标

你需要完成：

1. 读取25,000条淘宝商品记录；
2. 检查字段类型和缺失值；
3. 完成列选择、行选择、条件筛选和排序；
4. 完成商品价格及一级品类统计；
5. 完成“省份—类别”挑战分析；
6. 输出两张统计表并撰写有边界的结论。

### 数据边界

- 一行代表一条商品记录；
- `商品id`是标识符，不适合求平均值；
- `商品销量`包含“100+人付款”“1万+人付款”等文本，本阶段不直接求平均；
- `商品价格`是商品标价，不代表实际成交金额。

## 任务0：环境与个人配置

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


STUDENT_NAME = "吴泽昊"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "淘宝全品类全国数据.csv").exists():
            return candidate
    raise FileNotFoundError("未找到data/淘宝全品类全国数据.csv")


ROOT = find_project_root()
DATA_PATH = ROOT / "data" / "淘宝全品类全国数据.csv"
OUTPUT_DIR = ROOT / "output" / "day03_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


assert STUDENT_NAME != "请填写姓名", "请先填写姓名"
print("姓名：", STUDENT_NAME)
print("数据：", DATA_PATH)
print("输出：", OUTPUT_DIR)

姓名： 吴泽昊
数据： D:\实训\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\data\淘宝全品类全国数据.csv
输出： D:\实训\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day03_analysis


## 任务1：读取数据并完成初步观察

In [2]:
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
print("字段名：", df.columns.tolist())
display(df.head())

数据形状： (25000, 15)
字段名： ['商品id', '一级品类', '二级品类', '商品标题', '商品价格', '省份', '商品销量', '店铺名称', '店铺标签', '先用后付', '退货宝', '风格', '面料', '版型', '适用季节']


,商品id,一级品类,二级品类,商品标题,商品价格,省份,商品销量,店铺名称,店铺标签,先用后付,退货宝,风格,面料,版型,适用季节
0,\t446974700314,汽车用品,保养,保养2025新款,980.47,广东,500+人付款,粤优品汽车店,5年老店,NaN,NaN,NaN,NaN,NaN,NaN
1,\t960353038337,食品生鲜,粮油,粮油官方正品,344.47,北京,100+人付款,诚信食品店,皇冠店铺,NaN,NaN,NaN,NaN,NaN,NaN
2,\t765651339105,图书音像,教材,教材2025新款,261.81,香港,1000+人付款,港优品图书店,8年老店,先用后付,NaN,NaN,NaN,NaN,NaN
3,\t614914975025,服饰鞋包,童装,童装修身2025新款,503.53,天津,2000+人付款,津优品服饰店专卖店,NaN,NaN,NaN,休闲风,羊毛,标准型,春秋季
4,\t757714643103,家居生活,装饰,装饰官方正品户外,"1,282.75",北京,500+人付款,时尚家居店旗舰店,回头客3千,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# 检查点1
assert df.shape == (25000, 15), "数据形状应为(25000, 15)"
assert {"商品id", "一级品类", "商品价格", "省份", "商品销量"}.issubset(df.columns)
print("检查点1通过")

检查点1通过


**数据粒度：** 请用一句话说明一行代表什么.

> TODO：一行代表一条独立的淘宝商品记录,包括商品id,一级品类,商品价格,省份,商品销量

## 任务2：字段类型与缺失值

In [4]:
# TODO 1：输出字段类型
print(df.dtypes)
# TODO 2：计算缺失数量并从高到低排序，变量名为missing_count
missing_count = df.isna().sum().sort_values(ascending=False)

# TODO 3：计算缺失率（百分比），变量名为missing_rate
missing_rate = (df.isna().sum()/len(df))*100

# TODO 4：展示结果
display(missing_count)
display(missing_rate)

商品id     object
一级品类     object
二级品类     object
商品标题     object
商品价格    float64
省份       object
商品销量     object
店铺名称     object
店铺标签     object
先用后付     object
退货宝      object
风格       object
面料       object
版型       object
适用季节     object
dtype: object


版型      22964
面料      22625
退货宝     22276
先用后付    21842
风格      21012
适用季节    20178
店铺标签     3741
商品销量        0
省份          0
商品价格        0
商品标题        0
二级品类        0
一级品类        0
商品id        0
店铺名称        0
dtype: int64

商品id    0.00
一级品类    0.00
二级品类    0.00
商品标题    0.00
商品价格    0.00
省份      0.00
商品销量    0.00
店铺名称    0.00
店铺标签   14.96
先用后付   87.37
退货宝    89.10
风格     84.05
面料     90.50
版型     91.86
适用季节   80.71
dtype: float64

In [5]:
# 检查点2
assert isinstance(missing_count, pd.Series), "missing_count应为Series"
assert isinstance(missing_rate, pd.Series), "missing_rate应为Series"
assert set(missing_count.index) == set(df.columns)
assert missing_count.sum() == df.isna().sum().sum()
print("检查点2通过")

检查点2通过


请填写：

- 一个可以直接进行数值统计的字段及原因；
- 一个暂不适合直接进行精确数值统计的字段及原因。

> TODO：可以直接进行数值统计的字段:商品价格 原因:它是数值型,可以直接计算均值和中位数等
不适合直接进行精确数值统计的字段:商品id 原因:这是标识符,无统计意义

## 任务3：选择列与选择行

In [6]:
# TODO 1：选择“商品价格”列，变量名为price_series
price_series = df["商品价格"]

# TODO 2：选择商品id、一级品类、商品价格、省份、商品销量五列
product_view = df[["商品id","一级品类","商品价格","省份","商品销量"]]

# TODO 3：分别使用loc和iloc取前5行局部数据
loc_view = df.iloc[:5]
iloc_view = df.loc[0:4]

# TODO 4：展示结果


In [7]:
# 检查点3
assert isinstance(price_series, pd.Series)
assert isinstance(product_view, pd.DataFrame)
assert product_view.shape == (25000, 5)
assert len(loc_view) == 5 and len(iloc_view) == 5
print("检查点3通过")

检查点3通过


请解释`df["商品价格"]`与`df[["商品价格"]]`的区别。

> TODO：前者返回一维数组，后者返回二维表

## 任务4：条件筛选与排序

In [8]:
# TODO 1：筛选广东商品，变量名为guangdong
guangdong = df[df["省份"] =="广东"]

# TODO 2：筛选广东且商品价格不低于1000元的商品
# 只保留商品id、一级品类、二级品类、商品价格、省份、商品销量
# 按商品价格从高到低排序，变量名为guangdong_high_price
cond = (df["省份"] == "广东") & (df["商品价格"] >= 1000)
guangdong_high_price = df[cond][["商品id", "一级品类", "二级品类", "商品价格", "省份", "商品销量"]].sort_values(by="商品价格", ascending=False)
# TODO 3：筛选浙江或江苏商品，变量名为zhejiang_or_jiangsu
zhejiang_or_jiangsu = df[df["省份"].isin(["浙江", "江苏"])]

# TODO 4：展示广东高价商品前10行和浙江或江苏商品数

In [9]:
# 检查点4
assert isinstance(guangdong, pd.DataFrame)
assert (guangdong["省份"] == "广东").all()
assert isinstance(guangdong_high_price, pd.DataFrame)
assert (guangdong_high_price["省份"] == "广东").all()
assert (guangdong_high_price["商品价格"] >= 1000).all()
assert guangdong_high_price["商品价格"].is_monotonic_decreasing
assert set(zhejiang_or_jiangsu["省份"].unique()).issubset({"浙江", "江苏"})
print("检查点4通过")

检查点4通过


## 任务5：描述性统计与一级品类汇总

In [10]:
# TODO 1：使用describe查看商品价格摘要，变量名为price_summary
price_summary = df["商品价格"].describe()

# TODO 2：按一级品类统计商品数、平均价格和中位价格
# 按平均价格从高到低排序，变量名为category_summary
category_summary = df.groupby("一级品类").agg(
    商品数=("商品id", "count"),
    平均价格=("商品价格", "mean"),
    中位价格=("商品价格", "median")
).sort_values(by="平均价格", ascending=False)

# TODO 3：展示结果

In [11]:
# 检查点5
assert isinstance(price_summary, pd.Series)
assert isinstance(category_summary, pd.DataFrame)
assert {"商品数", "平均价格", "中位价格"}.issubset(category_summary.columns)
assert category_summary["商品数"].sum() == len(df)
assert category_summary["平均价格"].is_monotonic_decreasing
assert abs(df["商品价格"].mean() - 938.26) < 0.02
print("检查点5通过")

检查点5通过


请写一条一级品类价格结论，并说明它不能代表什么。

> TODO：手机数码品类平均标价高，但是不代表成交价格，因为实际交易中存在优惠券

## 挑战任务：省份—类别分析

In [12]:
# TODO 1：选择两个省份
provinces = ["广东", "江苏"]

# TODO 2：筛选省份并统计商品数、平均价格和中位价格
province_data = df[df["省份"].isin(provinces)]
province_summary = province_data.groupby("省份").agg(
    商品数=("商品id", "count"),
    平均价格=("商品价格", "mean"),
    中位价格=("商品价格", "median")
)

# TODO 3：分别计算两个省份最常见的一级品类
top_categories = province_data.groupby(["省份", "一级品类"]).size().reset_index(name="count").sort_values(by=["省份", "count"], ascending=[True, False]).groupby("省份").head(1)

# TODO 4：展示结果

In [13]:
# 检查点6
assert len(provinces) == 2 and provinces[0] != provinces[1]
assert isinstance(province_summary, pd.DataFrame)
assert set(province_summary.index) == set(provinces)
assert {"商品数", "平均价格", "中位价格"}.issubset(province_summary.columns)
assert isinstance(top_categories, pd.DataFrame)
print("检查点6通过")

检查点6通过


## 输出成果

In [14]:
outputs = {
    "category_summary.csv": category_summary.reset_index(),
    "province_summary.csv": province_summary.reset_index(),
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    reloaded = pd.read_csv(path)
    assert reloaded.shape == table.shape
    assert not any(str(col).startswith("Unnamed") for col in reloaded.columns)
    print("已输出：", path.relative_to(ROOT))

已输出： output\day03_analysis\category_summary.csv
已输出： output\day03_analysis\province_summary.csv


## 结论与边界

请至少完成两条结论，每条包含：数据范围、字段与方法、数据结论、结论边界。

### 结论1

> TODO：价格分层极其显著，数码家电具有绝对溢价：数码家电的平均价格（3085.53元）和中位价格（3116.12元）均遥遥领先，是第二名钟表珠宝（均价1981.20元）的1.5倍以上，属于典型的高客单价品类。

### 结论2

> TODO：广东是核心供给大省：广东省的商品数量（2303件）显著高于江苏省（1763件），多出约30.6%，是平台最主要的货源地/商家聚集地。

### GitHub提交检查

- [ ] Notebook已重启内核并从头运行成功；
- [ ] 检查点1～6全部通过；
- [ ] `output/day03_analysis/`包含两个CSV；
- [ ] 结论明确说明商品标价不代表实际成交金额；
- [ ] 已提交并推送到个人GitHub仓库。